In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

#### LOAD ROLES

In [ ]:
roles = pd.read_csv("../data/rolle.tsv", sep="\t")
roles.loc[len(roles)] = [-1, 'UNKNOWN']
roles.loc[len(roles)] = [-0, 'REGIE']
roles = roles.rename(columns={'rolleID': 'speaker_id'})
roles = roles.set_index('speaker_id')
roles

#### LOAD GENDER

In [ ]:
# load gender
gender_df = pd.read_csv('../data/gender/gender.csv')
gender_df = gender_df.rename(columns={'rolleID': 'speaker_id'})
gender_df

#### LOAD COMBINED SCRIPTS

In [ ]:
scripts = pd.read_csv('./combine.csv')
scripts = scripts.drop(columns=['Text','Page', 'Episode'])
scripts

#### GROUP SCRIPT LINES BY SPEAKER ID AND COUNT

In [ ]:
speaker_counts = scripts.groupby(by='speaker_id').count()
speaker_counts = speaker_counts.rename(columns={'Speaker': 'count'})
speaker_counts

#### JOIN AND ADD REAL NAMES

In [ ]:
speaker_counts = speaker_counts.join(roles, how='left', on='speaker_id')
speaker_counts = speaker_counts.sort_values(by='count', ascending=False)
speaker_counts = speaker_counts.reset_index()
speaker_counts

#### ADD GENDER TO SPEAKER COUNTS

In [ ]:
speaker_counts_gender = pd.merge(speaker_counts, gender_df, on='speaker_id', how='left')
speaker_counts_gender.head(15)

#### VISUALISE TOP 15 SPEAKER

In [ ]:
gender_colors = {'m': 'orange', 'f': 'skyblue', np.nan: 'black'}

fig, ax = plt.subplots(figsize=(10, 6))

for i, row in speaker_counts_gender.head(15).iterrows():
    ax.barh(row['name'], row['count'], color=gender_colors[row['gender']])

plt.title('Number of Spoken Lines in all Audiobooks by Speaker')
plt.xlabel('Number of Spoken Lines')
plt.ylabel('Speaker')
plt.show()

#### LIST NUMBER OF SPOKEN LINES PER GENDER

In [ ]:
spoken_lines_by_gender = speaker_counts_gender.groupby(by='gender')['count'].sum()
spoken_lines_by_gender

#### VISUALISE NUMBER OF SPOKEN LINES PER GENDER

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

ax.barh('F', list(spoken_lines_by_gender)[0], color='skyblue')
ax.barh('M', list(spoken_lines_by_gender)[1], color='orange')

plt.title('Number of Spoken Lines per Gender')
plt.xlabel('Number of Spoken Lines')
plt.ylabel('Gender')
plt.show()

#### LIST GENDER UNKNOW, SORTED BY NUMBER OF LINES

In [ ]:
unknown_gender = speaker_counts_gender[speaker_counts_gender['gender'].isna()].sort_values('count', ascending=False)
unknown_gender

#### List names UNKNOWN

In [ ]:
unknowns = scripts.where(
    scripts['speaker_id']==-1
).groupby(
    by='Speaker'
).count().rename(
    columns={'speaker_id' : 'count'}
).sort_values(by='count', ascending=False)
unknowns